Imports and Setup

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

load_dotenv()
print('✅ Imports ready')

✅ Imports ready


Load the sample document

In [2]:
with open('../../data/sample_crops.txt', 'r', encoding='utf-8') as f:
    text = f.read()
    
print(text)

Cassava Mosaic Disease is a viral disease affecting cassava plants. Symptoms include yellowing and mottling of leaves, stunted growth, and reduced yield. It is spread by whiteflies and through infected cuttings. Control methods include using disease-free cuttings, planting resistant varieties, and removing infected plants.

Maize Smut is a fungal disease that affects maize. Symptoms include large, swollen galls on cobs, leaves, and stalks. Control methods include planting resistant hybrids, crop rotation, and removing galls before they burst.

Rice Blast is a fungal disease of rice. Symptoms include diamond-shaped lesions on leaves and neck rot. Control methods include using resistant varieties, balanced nitrogen fertilizer, and fungicide application when needed.

Tomato Leaf Curl Virus affects tomato plants. Symptoms include upward curling of leaves, yellowing, and stunted growth. It is spread by whiteflies. Control methods include using resistant varieties, controlling whiteflies, an

Split the document into chunks

In [3]:
doc = Document(page_content=text, metadata={'source': 'sample_crops.txt'})

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=0,
    separators=['\n\n']    # Split ONLY at blank lines
)

chunks = splitter.split_documents([doc])

print(f'Total chunks: {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'--- Chunk {i} ---')
    print(c.page_content)
    print()

Total chunks: 3
--- Chunk 0 ---
Cassava Mosaic Disease is a viral disease affecting cassava plants. Symptoms include yellowing and mottling of leaves, stunted growth, and reduced yield. It is spread by whiteflies and through infected cuttings. Control methods include using disease-free cuttings, planting resistant varieties, and removing infected plants.

--- Chunk 1 ---
Maize Smut is a fungal disease that affects maize. Symptoms include large, swollen galls on cobs, leaves, and stalks. Control methods include planting resistant hybrids, crop rotation, and removing galls before they burst.

Rice Blast is a fungal disease of rice. Symptoms include diamond-shaped lesions on leaves and neck rot. Control methods include using resistant varieties, balanced nitrogen fertilizer, and fungicide application when needed.

--- Chunk 2 ---
Tomato Leaf Curl Virus affects tomato plants. Symptoms include upward curling of leaves, yellowing, and stunted growth. It is spread by whiteflies. Control metho

Embed the chunks and store them in a vector store

In [4]:
# Load the embedding model (converts text to vectors)
embeddings = OpenAIEmbeddings()

# Build an in-memory Chroma store from chunks
vectorstore = Chroma.from_documents(chunks, embedding=embeddings)

print('Vector store built')

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store built


Inspect the vector store

In [5]:
# Number of vectors stored
print(f'Total vectors: {vectorstore._collection.count()}')

# Fetch everything from the store, INCLUDING embeddings
data = vectorstore.get(include=['documents', 'embeddings', 'metadatas'])

print(f'\nIDs: {data["ids"]}')

print('\nNumber of documents:', len(data['documents']))
print('\nDocuments:')
for i, doc in enumerate(data['documents']):
    print(f'  [{i}] {doc[:80]} ...')

print('\nNumber of embeddings:', len(data['embeddings']))
print('Embedding dimension:', len(data['embeddings'][0]))

print('\nEmbeddings (first 5 numbers of each):')
for i, emb in enumerate(data['embeddings']):
    print(f'  [{i}] {emb[:5]}')

Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Total vectors: 3

IDs: ['15247942-c0c6-41cc-8d2c-44009b95c6c3', '74e9f5ec-23b0-4cb4-a478-3a14118d2cce', 'b6328cef-6f03-43c2-815a-64832a5e4a86']

Number of documents: 3

Documents:
  [0] Maize Smut is a fungal disease that affects maize. Symptoms include large, swoll ...
  [1] Tomato Leaf Curl Virus affects tomato plants. Symptoms include upward curling of ...
  [2] Cassava Mosaic Disease is a viral disease affecting cassava plants. Symptoms inc ...

Number of embeddings: 3
Embedding dimension: 1536

Embeddings (first 5 numbers of each):
  [0] [-0.019116459414362907, -0.013523026369512081, 0.02942226082086563, 0.006624680943787098, -0.02509702928364277]
  [1] [-0.0009174961596727371, -0.025549115613102913, 0.003800938604399562, -0.025405067950487137, -0.015112086199223995]
  [2] [-0.023043060675263405, -0.005587385967373848, 0.018947388976812363, -0.017063118517398834, -0.01677524298429489]


Plain retrieval (no history)

In [6]:
# Retriever returns top 2 closest chunks
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})

# A clear, standalone question
query = 'How can I control Cassava Mosaic Disease?'

#  Search the vector store
results = retriever.invoke(query)

print(f'Query: {query}\n')
for i, doc in enumerate(results):
    print(f'--- Result {i} ---')
    print(doc.page_content)
    print()

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Query: How can I control Cassava Mosaic Disease?

--- Result 0 ---
Cassava Mosaic Disease is a viral disease affecting cassava plants. Symptoms include yellowing and mottling of leaves, stunted growth, and reduced yield. It is spread by whiteflies and through infected cuttings. Control methods include using disease-free cuttings, planting resistant varieties, and removing infected plants.

--- Result 1 ---
Maize Smut is a fungal disease that affects maize. Symptoms include large, swollen galls on cobs, leaves, and stalks. Control methods include planting resistant hybrids, crop rotation, and removing galls before they burst.

Rice Blast is a fungal disease of rice. Symptoms include diamond-shaped lesions on leaves and neck rot. Control methods include using resistant varieties, balanced nitrogen fertilizer, and fungicide application when needed.



In [7]:
query = 'How can I control the first one?'    # Vague — "the first one" means nothing alone
results = retriever.invoke(query)              # Search vector store

print(f'Query: {query}\n')
for i, doc in enumerate(results):
    print(f'--- Result {i} ---')
    print(doc.page_content)
    print()

Query: How can I control the first one?

--- Result 0 ---
Maize Smut is a fungal disease that affects maize. Symptoms include large, swollen galls on cobs, leaves, and stalks. Control methods include planting resistant hybrids, crop rotation, and removing galls before they burst.

Rice Blast is a fungal disease of rice. Symptoms include diamond-shaped lesions on leaves and neck rot. Control methods include using resistant varieties, balanced nitrogen fertilizer, and fungicide application when needed.

--- Result 1 ---
Tomato Leaf Curl Virus affects tomato plants. Symptoms include upward curling of leaves, yellowing, and stunted growth. It is spread by whiteflies. Control methods include using resistant varieties, controlling whiteflies, and removing infected plants.



Create the LLM and the rewrite prompt

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)          

rewrite_prompt = ChatPromptTemplate.from_messages([
    ('system', 'Given a chat history and the latest user question '           # Instruction: rewrite, don\'t answer
     'which might reference context in the chat history, '
     'formulate a standalone question which can be understood '
     'without the chat history. Do NOT answer the question, '
     'just reformulate it if needed and otherwise return it as is.'),
    MessagesPlaceholder('chat_history'),
    ('human', '{input}'),
])

print('LLM and rewrite prompt ready')

LLM and rewrite prompt ready


See the rewrite in action

In [9]:
from langchain_core.messages import HumanMessage, AIMessage

chat_history = [
    HumanMessage(content='What are the common crop diseases?'),
    AIMessage(content='Common crop diseases include Cassava Mosaic Disease, '           # Assistant's reply
                      'Maize Smut, Rice Blast, and Tomato Leaf Curl Virus.'),
]

vague_question = 'How can I control the first one?'

# Build a simple chain: rewrite prompt → LLM
rewrite_chain = rewrite_prompt | llm

# Invoke with the history and the vague question
rewritten = rewrite_chain.invoke({
    'chat_history': chat_history,
    'input': vague_question
})

print('Original question:', vague_question)
print('Rewritten question:', rewritten.content)

Original question: How can I control the first one?
Rewritten question: What methods can be used to control Cassava Mosaic Disease?


Retrieve using the rewritten question

In [10]:
rewritten_query = rewritten.content
print('Rewritten query')

# Search the vector store
results = retriever.invoke(rewritten_query)

print(f'Rewritten query: {rewritten_query}\n')

for i, doc in enumerate(results):
    print(f'--- Result {i} ---')
    print(doc.page_content)
    print()

Rewritten query
Rewritten query: What methods can be used to control Cassava Mosaic Disease?

--- Result 0 ---
Cassava Mosaic Disease is a viral disease affecting cassava plants. Symptoms include yellowing and mottling of leaves, stunted growth, and reduced yield. It is spread by whiteflies and through infected cuttings. Control methods include using disease-free cuttings, planting resistant varieties, and removing infected plants.

--- Result 1 ---
Maize Smut is a fungal disease that affects maize. Symptoms include large, swollen galls on cobs, leaves, and stalks. Control methods include planting resistant hybrids, crop rotation, and removing galls before they burst.

Rice Blast is a fungal disease of rice. Symptoms include diamond-shaped lesions on leaves and neck rot. Control methods include using resistant varieties, balanced nitrogen fertilizer, and fungicide application when needed.



Generate the final answer using retrieved context

In [11]:
from langchain_core.prompts import ChatPromptTemplate

# QA prompt: instructions + context + question
qa_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a helpful assistant. '                              # Instruction
     'Answer the question using only the provided context. '
     'If you don\'t know, say you don\'t know.'),
    ('human', 'Context:\n{context}\n\nQuestion: {input}') # Context + question slots
])

# Join all retrieved chunks into one text
context = '\n\n'.join(doc.page_content for doc in results)  

# Chain: prompt → LLM
qa_chain = qa_prompt | llm

final_answer = qa_chain.invoke({
    'context': context,
    'input': rewritten_query
})

print('Question:', rewritten_query)
print('\nAnswer:')
print(final_answer.content)

Question: What methods can be used to control Cassava Mosaic Disease?

Answer:
Control methods for Cassava Mosaic Disease include using disease-free cuttings, planting resistant varieties, and removing infected plants.


Using LangChain helpers

 **create_history_aware_retriever** (for rewrite + retrieve) 

**create_stuff_documents_chain** (for context + answer).

In [12]:
from langchain.chains import create_history_aware_retriever
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnablePassthrough

Rewrite prompt and QA prompt

In [13]:
history_aware_retriever = create_history_aware_retriever(
    llm,
    retriever,
    rewrite_prompt
)

# QA step: takes docs + question, produces answer
qa_documents_chain = create_stuff_documents_chain(llm, qa_prompt)


Combine and run

In [14]:
# Combine the two chains together

rag_chain = (
    {'context': history_aware_retriever, 'input': RunnablePassthrough()}
    | qa_documents_chain
)

# Run it with the same vague question and history
result = rag_chain.invoke({
    'input': vague_question,
    'chat_history': chat_history,
})

print('Answer from compact chain:')
print(result)

Answer from compact chain:
To control Cassava Mosaic Disease, you can use disease-free cuttings, plant resistant varieties, and remove infected plants.


Create an empty store for session histories

In [15]:
store = {}  # Dictionary: session_id → chat history

 Define a function to fetch or create a session history

In [16]:
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    
    # If session doesn't exist...
    if session_id not in store:
        
        # create a new one
        store[session_id] = InMemoryChatMessageHistory()
        
    # Return the history for this session
    return store[session_id]

print('Session history function ready')

Session history function ready


Wrap the chain with automatic history management

In [17]:
from langchain_core.runnables import RunnableWithMessageHistory

conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,                 # Base RAG chain
    get_session_history,       # Function to fetch/create history
    input_messages_key='input',    # User query key
    history_messages_key='chat_history',   # Chat history key
    #output_messages_key='answer'   # Assistant response key
    
)

Test the wrapped chain with a multi‑turn conversation

In [18]:
session_id = 'demo-1'    # A unique ID for this conversation

# Turn 1 — no history yet
q1 = 'What are the common crop diseases?'
a1 = conversational_rag_chain.invoke(
    {'input': q1},
    config={'configurable': {'session_id': session_id}},
)

print('👤 User:', q1)
print('🤖 Assistant:', a1)
print('-' * 100)

👤 User: What are the common crop diseases?
🤖 Assistant: The common crop diseases mentioned are Maize Smut, Rice Blast, and Cassava Mosaic Disease.
----------------------------------------------------------------------------------------------------


Follow‑up question using history

In [19]:
q2 = 'How can I control the first one?'

a2 = conversational_rag_chain.invoke(
    {'input': q2},
    config={'configurable': {'session_id': session_id}},
)

print('👤 User:', q2)
print('🤖 Assistant:', a2)
print('-' * 100)

👤 User: How can I control the first one?
🤖 Assistant: To control Maize Smut, you can use the following methods: plant resistant hybrids, practice crop rotation, and remove galls before they burst.
----------------------------------------------------------------------------------------------------


In [20]:
q3 = 'How can I control the second one?'

a3 = conversational_rag_chain.invoke(
    {'input': q3},
    config={'configurable': {'session_id': session_id}},
)

print('👤 User:', q3)
print('🤖 Assistant:', a3)
print('-' * 100)

👤 User: How can I control the second one?
🤖 Assistant: To control Rice Blast, you can use the following methods: use resistant varieties, apply balanced nitrogen fertilizer, and use fungicides when needed.
----------------------------------------------------------------------------------------------------


In [21]:
# Turn 3 — clear follow-up, but still within the same session
q4 = 'What about Rice Blast?'
a4 = conversational_rag_chain.invoke(
    {'input': q4},
    config={'configurable': {'session_id': session_id}},
)

print('👤 User:', q4)
print('🤖 Assistant:', a4)
print('-' * 60)

👤 User: What about Rice Blast?
🤖 Assistant: To control Rice Blast, you can use the following methods: use resistant varieties, apply balanced nitrogen fertilizer, and use fungicides when needed.
------------------------------------------------------------


In [22]:
q5 = 'What about the last one?'

a5 = conversational_rag_chain.invoke(
    {'input': q5},
    config={'configurable': {'session_id': session_id}},
)

print('👤 User:', q5)
print('🤖 Assistant:', a5)
print('-' * 100)

👤 User: What about the last one?
🤖 Assistant: To control Rice Blast, you can use the following methods: use resistant varieties, apply balanced nitrogen fertilizer, and use fungicides when needed.
----------------------------------------------------------------------------------------------------


In [23]:
q6 = 'What about the first one again?'

a6 = conversational_rag_chain.invoke(
    {'input': q6},
    config={'configurable': {'session_id': session_id}},
)

print('👤 User:', q6)
print('🤖 Assistant:', a6)
print('-' * 100)

👤 User: What about the first one again?
🤖 Assistant: To control Maize Smut, you can use the following methods: plant resistant hybrids, practice crop rotation, and remove galls before they burst.
----------------------------------------------------------------------------------------------------


 Inspect the saved chat history (optional but instructive)

In [24]:
# Show all messages stored for this session

history = store[session_id].messages

print(f'Total messages in session "{session_id}": {len(history)}\n')

for i, msg in enumerate(history):
    role = '👤 User' if msg.type == 'human' else '🤖 Assistant'
    print(f'[{i}] {role}: {msg.content[:120]}')
    print()

Total messages in session "demo-1": 12

[0] 👤 User: What are the common crop diseases?

[1] 🤖 Assistant: The common crop diseases mentioned are Maize Smut, Rice Blast, and Cassava Mosaic Disease.

[2] 👤 User: How can I control the first one?

[3] 🤖 Assistant: To control Maize Smut, you can use the following methods: plant resistant hybrids, practice crop rotation, and remove ga

[4] 👤 User: How can I control the second one?

[5] 🤖 Assistant: To control Rice Blast, you can use the following methods: use resistant varieties, apply balanced nitrogen fertilizer, a

[6] 👤 User: What about Rice Blast?

[7] 🤖 Assistant: To control Rice Blast, you can use the following methods: use resistant varieties, apply balanced nitrogen fertilizer, a

[8] 👤 User: What about the last one?

[9] 🤖 Assistant: To control Rice Blast, you can use the following methods: use resistant varieties, apply balanced nitrogen fertilizer, a

[10] 👤 User: What about the first one again?

[11] 🤖 Assistant: To control Maize